## Building RAG System using LlamaIndex

##### https://www.llamaindex.ai/

1. Data INGESTION (data loading, chunking+nodes, embedding)
2. Indexing
3. Retriever
4. Response Synthesizer
5. Querying

In [103]:
#!pip install llama-index

In [101]:
# install the openai api key
from getpass import getpass
OPENAI_API_KEY =  getpass("Enter Your OpenAI API Key:")

Enter Your OpenAI API Key: ········


# Step 1 - Data Ingestion
### Data Loaders

In [106]:
from llama_index.core import SimpleDirectoryReader

In [110]:
documents = SimpleDirectoryReader(input_files = ["data\\transformer_paper.pdf"]).load_data()

In [112]:
len(documents)

15

In [114]:
documents[0]

Document(id_='ce729424-5625-4428-b3d0-9b54081c1739', embedding=None, metadata={'page_label': '1', 'file_name': 'transformer_paper.pdf', 'file_path': 'data\\transformer_paper.pdf', 'file_type': 'application/pdf', 'file_size': 2215244, 'creation_date': '2025-07-26', 'last_modified_date': '2025-07-22'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.

In [118]:
documents[0].metadata

{'page_label': '1',
 'file_name': 'transformer_paper.pdf',
 'file_path': 'data\\transformer_paper.pdf',
 'file_type': 'application/pdf',
 'file_size': 2215244,
 'creation_date': '2025-07-26',
 'last_modified_date': '2025-07-22'}

In [120]:
documents[0].text

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurren

# Chunking

In [123]:
from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter

In [129]:
splitter = SentenceSplitter(chunk_size=500,chunk_overlap=20)
nodes = splitter.get_nodes_from_documents(documents)

In [131]:
print(len(documents))
print(len(nodes))

15
29


In [134]:
splitter

SentenceSplitter(include_metadata=True, include_prev_next_rel=True, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x00000186F1CB2840>, id_func=<function default_id_func at 0x00000186ED89B9C0>, chunk_size=500, chunk_overlap=20, separator=' ', paragraph_separator='\n\n\n', secondary_chunking_regex='[^,.;。？！]+[,.;。？！]?|[,.;。？！]')

In [136]:
nodes

[TextNode(id_='5e80ccfb-dc8a-4912-a0c6-c0f848b5b653', embedding=None, metadata={'page_label': '1', 'file_name': 'transformer_paper.pdf', 'file_path': 'data\\transformer_paper.pdf', 'file_type': 'application/pdf', 'file_size': 2215244, 'creation_date': '2025-07-26', 'last_modified_date': '2025-07-22'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ce729424-5625-4428-b3d0-9b54081c1739', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': 'transformer_paper.pdf', 'file_path': 'data\\transformer_paper.pdf', 'file_type': 'application/pdf', 'file_size': 2215244, 'creation_date': '2025-07-26', 'last_modified_date': '2025-07-22'}, hash='cf75942ca75bd7909aaf66a07a318b4db070ec76fc164d27c

# Embedding Model

In [152]:
from llama_index.embeddings.openai import OpenAIEmbedding
embed_model = OpenAIEmbedding(api_key = OPENAI_API_KEY, model="text-embedding-3-small")

# Indexing / VectorStorage

In [154]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex.from_documents(nodes, embed_model=embed_model)

In [156]:
index

# Retrieval

In [159]:
from llama_index.llms.openai import OpenAI
llm = OpenAI(api_key = OPENAI_API_KEY, model="gpt-4o-mini")

In [161]:
# setting up the index as Retriever
retriever = index.as_retriever()

In [163]:
# Retrive information based on the query 
retrieved_nodes = retriever.retrieve("What is Transformers?")
retrieved_nodes

[NodeWithScore(node=TextNode(id_='e0342b7f-7acd-4e6e-b7d5-c2f6eee32e90', embedding=None, metadata={'page_label': '2', 'file_name': 'transformer_paper.pdf', 'file_path': 'data\\transformer_paper.pdf', 'file_type': 'application/pdf', 'file_size': 2215244, 'creation_date': '2025-07-26', 'last_modified_date': '2025-07-22'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='d71118b8-725d-4b49-b2ad-498464f71675', node_type='4', metadata={'page_label': '2', 'file_name': 'transformer_paper.pdf', 'file_path': 'data\\transformer_paper.pdf', 'file_type': 'application/pdf', 'file_size': 2215244, 'creation_date': '2025-07-26', 'last_modified_date': '2025-07-22'}, hash='31a01c6ac9d6e719d6902c205c764b6ed976ab27d40c88c29cf6c

In [169]:
retrieved_nodes[0].metadata

{'page_label': '2',
 'file_name': 'transformer_paper.pdf',
 'file_path': 'data\\transformer_paper.pdf',
 'file_type': 'application/pdf',
 'file_size': 2215244,
 'creation_date': '2025-07-26',
 'last_modified_date': '2025-07-22'}

In [171]:
retrieved_nodes[1].metadata

{'page_label': '10',
 'file_name': 'transformer_paper.pdf',
 'file_path': 'data\\transformer_paper.pdf',
 'file_type': 'application/pdf',
 'file_size': 2215244,
 'creation_date': '2025-07-26',
 'last_modified_date': '2025-07-22'}

In [173]:
retrieved_nodes[2].metadata

IndexError: list index out of range

In [175]:
retrieved_nodes = retriever.retrieve("What is self attention mechanism?")
retrieved_nodes

[NodeWithScore(node=TextNode(id_='670b8087-1c94-4d4b-b751-5fbc92a50922', embedding=None, metadata={'page_label': '6', 'file_name': 'transformer_paper.pdf', 'file_path': 'data\\transformer_paper.pdf', 'file_type': 'application/pdf', 'file_size': 2215244, 'creation_date': '2025-07-26', 'last_modified_date': '2025-07-22'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ef53ec27-91fc-4915-9326-ad8c24243721', node_type='4', metadata={'page_label': '6', 'file_name': 'transformer_paper.pdf', 'file_path': 'data\\transformer_paper.pdf', 'file_type': 'application/pdf', 'file_size': 2215244, 'creation_date': '2025-07-26', 'last_modified_date': '2025-07-22'}, hash='9ab44c889c0af5e8205396b428f13cc4c450b5c49af2be55831cb

In [177]:
retrieved_nodes[0].text

'4 Why Self-Attention\nIn this section we compare various aspects of self-attention layers to the recurrent and convolu-\ntional layers commonly used for mapping one variable-length sequence of symbol representations\n(x1, ..., xn) to another sequence of equal length (z1, ..., zn), with xi, zi ∈ Rd, such as a hidden\nlayer in a typical sequence transduction encoder or decoder. Motivating our use of self-attention we\nconsider three desiderata.\nOne is the total computational complexity per layer. Another is the amount of computation that can\nbe parallelized, as measured by the minimum number of sequential operations required.\nThe third is the path length between long-range dependencies in the network. Learning long-range\ndependencies is a key challenge in many sequence transduction tasks. One key factor affecting the\nability to learn such dependencies is the length of the paths forward and backward signals have to\ntraverse in the network. The shorter these paths between any combin

In [179]:
retrieved_nodes[1].text

'This makes\nit more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\nused successfully in a variety of tasks including reading comprehension, abstractive summarization,\ntextual entailment and learning task-independent sentence representations [4, 27, 28, 22].\nEnd-to-end memory networks are based on a recurrent attention mechanism instead of sequence-\naligned recurrence and have been shown to perform well on simple-language question answering and\nlanguage modeling tasks [34].\nTo the best of our knowledge, however, th

In [181]:
retrieved_nodes[2].text

IndexError: list index out of range

# Response Synthesizer

In [186]:
from llama_index.core import get_response_synthesizer
response_synthesizer = get_response_synthesizer(llm=llm)

# Querying

In [189]:
query_engine = index.as_query_engine(llm=llm, response_synthesizer=response_synthesizer)
response = query_engine.query("What is Transformers?")
display(Markdown(response.response))

Transformers are a type of neural network architecture designed for sequence transduction tasks, which rely entirely on self-attention mechanisms to compute representations of input and output sequences. Unlike traditional models that use recurrent neural networks (RNNs) or convolutional layers, Transformers utilize self-attention to relate different positions within a single sequence, allowing for efficient processing of dependencies across distant positions. This architecture typically consists of an encoder-decoder structure, where the encoder transforms an input sequence into continuous representations, and the decoder generates an output sequence in an auto-regressive manner. Transformers have been successfully applied to various tasks, including machine translation, reading comprehension, and summarization.

In [191]:
query_engine = index.as_query_engine(llm=llm, response_synthesizer=response_synthesizer)
response = query_engine.query("What is self attention and how the architecture looks like, explain in details with step by step approach? Also, use simple language so that everyone can understand")
display(Markdown(response.response))

Self-attention, also known as intra-attention, is a mechanism that allows a model to relate different positions within a single sequence to compute a representation of that sequence. This means that when processing a sequence of words or symbols, the model can consider the importance of each word in relation to every other word, regardless of their position in the sequence.

### Step-by-Step Explanation of Self-Attention:

1. **Input Representation**: 
   - The input sequence is first transformed into a set of vectors. Each word or symbol in the sequence is represented as a vector, capturing its meaning.

2. **Calculating Attention Scores**:
   - For each word in the sequence, the model calculates how much attention it should pay to every other word. This is done by taking the dot product of the vector of the current word with the vectors of all other words. The result is a score that indicates the relevance of each word to the current word.

3. **Applying Softmax**:
   - The attention scores are then passed through a softmax function. This converts the scores into probabilities, ensuring that they sum up to 1. Higher scores indicate that the model should pay more attention to those words.

4. **Weighted Sum of Vectors**:
   - Each word's vector is then multiplied by its corresponding attention score (the probability from the softmax). This creates a weighted representation of the sequence, where more important words have a greater influence.

5. **Output Representation**:
   - Finally, the weighted vectors are summed up to produce a single output vector for the current word. This output vector now contains information from the entire sequence, emphasizing the most relevant parts.

### Architecture Overview:

- **Encoder-Decoder Structure**: 
   - The self-attention mechanism is typically used within an encoder-decoder architecture. The encoder processes the input sequence and generates a set of continuous representations. The decoder then uses these representations to generate an output sequence, one element at a time.

- **Parallelization**:
   - One of the advantages of self-attention is that it allows for parallel processing of the input sequence. Unlike recurrent layers, which process data sequentially, self-attention can compute all attention scores simultaneously, making it faster and more efficient.

- **Handling Long-Range Dependencies**:
   - Self-attention helps in learning long-range dependencies effectively. Since every word can directly attend to every other word, the model can easily capture relationships between distant words in the sequence.

In summary, self-attention is a powerful mechanism that enhances the model's ability to understand and represent sequences by allowing it to focus on relevant parts of the input, regardless of their position. This leads to improved performance in various tasks such as reading comprehension and summarization.